# Break Through Tech AI: Nestlé 1A Group
## Stage 1: Building the DataFrame

As of 09/06/2025, the Nestlé 1A group has decided to use a subset of the [Amazon Reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) dataset collected in 2023 by McAuley Lab. The entire dataset consists of 571.54 M examples. We are using the raw data from the "Grocery and Gourmet Food" category, which consists of over 14 M examples, to build an initial dataframe to then preprocess and develop machine learning models from.

Note: Outputs have been cleared due to rendering issues. Please run on your personal machine.

#### Step 0. Update, Install, and Import Python Libraries

In [1]:
#%pip install --upgrade pip
#%pip install -q datasets huggingface_hub pyarrow pandas
#%pip install matplotlib
#%pip install seaborn
#%pip install scikit-learn
#%pip install seaborn

In [2]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from tqdm import tqdm
import gc
import json
import pandas as pd
import pyarrow as pa

#### Step 1. Upload User Reviews file from HuggingFace

In [3]:
REV_PATH = "raw/review_categories/Grocery_and_Gourmet_Food.jsonl"

In [4]:
rev_file = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023", filename=REV_PATH, repo_type="dataset",)

In [5]:
ds_rev = load_dataset("json", data_files=rev_file, split="train")

In [6]:
# Note: This cell may take a while to run.
df_rev = ds_rev.to_pandas()

##### Data Fields for User Reviews

| Field            | Type   | Explanation |
| :--------------- | :----- | :---------- |
| rating           | float  | Rating of the product (from 1.0 to 5.0). |
| text             | str    | Text body of the user review. |
| images           | list   | Images that users post after they have received the product.<br><br>Note: Each image has different sizes (small, medium, large), represented by the `small_image_url`, `medium_image_url`, and `large_image_url` respectively. |
| asin             | str    | ID of the product. |
| parent_asin      | str    | Parent ID of the product.<br><br>Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta. |
| user_id          | str    | ID of the reviewer. |
| timestamp        | int    | Time of the review (unix time). |
| verified_purchase| bool   | User purchase verification. |
| helpful_vote     | int    | Helpful votes of the review. |

#### Step 2. Upload Item Metadata file from HuggingFace

In [7]:
META_PATH = "raw/meta_categories/meta_Grocery_and_Gourmet_Food.jsonl"

In [8]:
# Filters down to only products with reviews
needed_parent_asin = set(df_rev["parent_asin"].unique())

In [9]:
# Loading metadata
meta_file = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename=META_PATH,
    repo_type="dataset",
)


In [10]:
# Selected few columns from metadata
meta_rows = []
with open(meta_file, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Parsing meta file"):
        obj = json.loads(line)
        row = {
            "parent_asin": obj.get("parent_asin"),
            "title_meta": obj.get("title"),
            "main_category": obj.get("main_category"),
            "average_rating": obj.get("average_rating"),
            "rating_number": obj.get("rating_number"),
            "price": obj.get("price"),
            "details": obj.get("details")  # keep raw JSON dict for now
        }
        # Filters down to only products with reviews
        if row["parent_asin"] not in needed_parent_asin:
            continue
        meta_rows.append(row)

Parsing meta file: 603274it [00:06, 94617.65it/s] 


In [11]:
df_meta = pd.DataFrame(meta_rows)

##### Data Fields for Item Metadata

| Field           | Type  | Explanation |
| :--------------- | :---- | :---------- |
| main_category    | str   | Main category (i.e., domain) of the product. |
| title            | str   | Name of the product. |
| average_rating   | float | Rating of the product shown on the product page. |
| rating_number    | int   | Number of ratings in the product. |
| features         | list  | Bullet-point format features of the product. |
| description      | list  | Description of the product. |
| price            | float | Price in US dollars (at time of crawling). |
| images           | list  | Images of the product. Each image has different sizes (thumb, large, hi_res). The “variant” field shows the position of the image. |
| videos           | list  | Videos of the product, including title and URL. |
| store            | str   | Store name of the product. |
| categories       | list  | Hierarchical categories of the product. |
| details          | dict  | Product details, including materials, brand, sizes, etc. |
| parent_asin      | str   | Parent ID of the product. |
| bought_together  | list  | Recommended bundles from the website. |


#### Step 3. Data Cleaning: Remove Duplicates & Handle Missing Values


##### Step 3a. User Reviews DataFrame

In [12]:
print("Shape before dropping duplicates:", df_rev.shape)

Shape before dropping duplicates: (14318520, 10)


In [13]:
# Check for duplicate reviews based on user_id, asin, and text
dup_count = df_rev.duplicated(subset=["user_id", "asin", "text"]).sum()
print("Total duplicate reviews:", dup_count)

Total duplicate reviews: 130960


In [14]:
# Drop duplicate reviews
df_rev = df_rev.drop_duplicates(subset=["user_id", "asin", "text"])
print("Shape after dropping duplicates:", df_rev.shape)

Shape after dropping duplicates: (14187560, 10)


In [15]:
# Count missing ratings
print("Missing ratings:", df_rev['rating'].isna().sum())

Missing ratings: 0


In [16]:
# Remove missing or empty review text
df_rev = df_rev.dropna(subset=['text'])
df_rev = df_rev[df_rev['text'].str.strip() != '']

In [17]:
# Count missing helpful_vote
print("Missing helpful_vote:", df_rev['helpful_vote'].isna().sum())

Missing helpful_vote: 0


In [18]:
# Count missing verified_purchase
print("Missing verified_purchase:", df_rev['verified_purchase'].isna().sum())

Missing verified_purchase: 0


In [19]:
print("Final dataset shape:", df_rev.shape)
print(df_rev.isna().sum())

Final dataset shape: (14170982, 10)
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64


##### Step 3b. Item Metadata DataFrame

In [20]:
print("Shape before dropping duplicates:", df_meta.shape)

Shape before dropping duplicates: (603182, 7)


In [21]:
print("Missing ratings:", df_meta['average_rating'].isna().sum())

Missing ratings: 0


#### Step 4. Merge the User Reviews and Item Metadata DataFrames into one

In [22]:
df = df_rev.merge(
    df_meta,
    on="parent_asin", how="left"
)

#### Step 5. Convert the file for Data Preprocessing

In [23]:
# Export merged dataframe as parquet chunks

chunk_size = 1000000  # 1M records per chunk
total_chunks = len(df) // chunk_size + 1

In [ ]:
for i in tqdm(range(0, len(df), chunk_size)):
    chunk = df.iloc[i:i+chunk_size]
    chunk['price'] = pd.to_numeric(df['price'].replace('—', np.nan), errors='coerce')

In [ ]:
chunk.to_parquet(f'df_chunk_{i//chunk_size:03d}.parquet', index=False)
    del chunk
    gc.collect()

In [ ]:
print(f"Exported {total_chunks} parquet chunks")